# SAC training on Kaggle — v2.4.2

**What changed in v2.4.2:**
- `target_entropy = -13` (was -65). Fixes the entropy-collapse failure observed in the v2.4.1 pilot.
- Optional WandB cloud sync — set `WANDB_API_KEY` as a Kaggle Secret to enable. Without it, training still runs (local-only logs).
- Rotating replay buffer kept (single file, ~3.1 GB) so the 2M-step run fits Kaggle's 20 GB output quota.

**Expected runtime:** ~8 hours per seed on Kaggle T4. Run as **Save Version → Save & Run All** to avoid browser-disconnect zombie sessions.

**To monitor live:** WandB dashboard at https://wandb.ai/<your-username>/sac-irrigation-thesis (URL is also printed in Cell 4's output).

In [ ]:
# CELL 1: Install + clone (run once per session)
import subprocess, sys, os, torch

# Clone latest repo
if os.path.exists('/kaggle/working/thesis'):
    subprocess.run(['rm', '-rf', '/kaggle/working/thesis'], check=True)
subprocess.run([
    'git', 'clone',
    'https://github.com/taratorbati/thesis.git',
    '/kaggle/working/thesis'
], check=True)

os.chdir('/kaggle/working/thesis')
sys.path.insert(0, '/kaggle/working/thesis')

# Install deps (wandb added for v2.4.2 cloud sync)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'casadi',
    'wandb>=0.16',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'PyTorch:           {torch.__version__}')
print(f'CUDA:              {torch.cuda.is_available()}')
try:
    import wandb
    print(f'wandb:             {wandb.__version__}')
except ImportError:
    print('wandb:             NOT INSTALLED')
torch.set_num_threads(4)
print('Setup complete.')

## Cell 2: Load WandB API key from Kaggle Secrets

**One-time setup (if not done yet):**
1. In the right sidebar of this notebook editor, click **Add-ons → Secrets**.
2. Click **Add a new secret**.
3. Label: `WANDB_API_KEY`. Value: paste the API key from https://wandb.ai/settings (under "API keys").
4. Toggle "Attached" on for this notebook.

If `WANDB_API_KEY` isn't set, training will still run — it just won't sync to the cloud.

In [ ]:
# CELL 2: Load WandB API key
import os
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = secret
    print('\u2713  WANDB_API_KEY loaded from Kaggle Secrets.')
    print('   WandB cloud sync will be active for this run.')
except Exception as e:
    print(f'\u26a0  Could not load WANDB_API_KEY from Kaggle Secrets ({type(e).__name__}).')
    print('   Training will run with local-only logging.')
    print('   To enable: Add-ons \u2192 Secrets \u2192 add WANDB_API_KEY.')

In [ ]:
# CELL 3: Smoke test (~2 min) — READ OUTPUT before running Cell 4
import time, numpy as np
from stable_baselines3 import SAC
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import CTDESACPolicy, make_sac_policy_kwargs
# Import train_sac early to catch any train.py syntax/import errors
# before burning hours on the real run.
from src.rl.train import train_sac

# IrrigationEnv() with no fixed_scenario: training mode (random year/budget)
env = IrrigationEnv(seed=0)
obs, _ = env.reset()
print(f'obs_dim:    {obs.shape[0]} (expected 707)')
print(f'action_dim: {env.action_space.shape[0]} (expected 130)')

model = SAC(
    policy=CTDESACPolicy, env=env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=10_000, batch_size=256,
    learning_starts=10, gradient_steps=1, verbose=0,
)
model.learn(total_timesteps=50, reset_num_timesteps=True)
t0 = time.time()
model.learn(total_timesteps=200, reset_num_timesteps=False)
rate = 200 / (time.time() - t0)
print(f'Device: {model.device}')
print(f'Rate:   {rate:.1f} steps/sec  ->  est {2_000_000/rate/3600:.1f} hrs for 2M steps')
print('PASSED' if obs.shape[0] == 707 else 'FAILED: wrong obs_dim')

In [ ]:
# CELL 4: Training
# Change SEED for each of the 5 parallel notebooks (0, 1, 2, 3, 4)
from src.rl.train import train_sac

SEED = 0  # use 0,1,2,3,4 across 5 parallel notebooks

train_sac(
    total_timesteps=2_000_000,
    seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    checkpoint_freq=50_000,
    eval_freq=25_000,
    wandb_project='sac-irrigation-thesis',
    verbose=1,
)

## What to watch on the WandB dashboard during training

Open the URL printed in Cell 4's output (e.g. https://wandb.ai/.../runs/...) and check:

- **`train/ent_coef`** — should decay gradually, NOT collapse below 0.01 in the first 25k steps. If it does, the entropy fix didn't hold; kill the run and fall back to fixed `ent_coef = 0.1` (set `hp_overrides={'ent_coef': 0.1}` in the Cell 4 call).
- **`rollout/ep_len_mean`** — should climb toward 93 (the season length). Values stuck near 60-70 mean the agent is burning the budget early.
- **`rollout/ep_rew_mean`** — should improve over time. Plateau is fine; sustained decline is not.
- **`eval/mean_reward`** — the headline number. Higher is better. Look for monotone-ish improvement.
- **System metrics** (left sidebar → System): GPU utilization should be >50% most of the time. RAM should plateau, not climb. Disk should stay flat (the rotating buffer doesn't grow).

In [ ]:
# CELL 5: Copy results to a clean output dir, skipping the giant replay buffer
import shutil, os

src = '/kaggle/working/thesis/results/rl'
dst = '/kaggle/working/results_rl'

# Clean up any previous attempts
if os.path.exists(dst):
    shutil.rmtree(dst)

# Skip replay buffer files (multi-GB) — model weights are small enough to keep.
def ignore_replay_buffers(dir, files):
    return [f for f in files if '_replay_buffer' in f or 'replay_buffer_latest' in f]

shutil.copytree(src, dst, ignore=ignore_replay_buffers)

total_mb = 0
for root, dirs, files in os.walk(dst):
    for f in files:
        path = os.path.join(root, f)
        mb = os.path.getsize(path) / 1e6
        total_mb += mb
        rel = os.path.relpath(path, dst)
        print(f'  {rel}  ({mb:.1f} MB)')

print(f'\nTotal: {total_mb:.0f} MB')

In [ ]:
# CELL 6: Zip + clickable download link
import shutil
from IPython.display import FileLink, display

shutil.make_archive('thesis_rl_models', 'zip', '/kaggle/working/results_rl')

print('Click the link below to download your models:')
display(FileLink('thesis_rl_models.zip'))

## Cell 7: Resume an interrupted run (DO NOT RUN on a fresh session)

If a previous run crashed or hit the time limit, you can resume from the latest checkpoint instead of starting over.

**Step-by-step:**
1. Find the previous run's `results/rl/sac_general_seed<N>/` directory in the failed notebook's Output.
2. Download the entire directory as a zip.
3. Create a Kaggle Dataset from it (Datasets → New Dataset → upload zip → publish private).
4. Add that Dataset as Input to this notebook (Add data → search for your dataset → Add).
5. Update `PREV_RESULTS` below to point at the dataset's path inside `/kaggle/input/`.
6. Update `CHECKPOINT_PATH` to point at the latest numbered checkpoint zip (e.g. `..._300000_steps.zip`).
7. Run Cell 1, Cell 2, then this cell.

**Important for v2.4.1+ runs:** Make sure the Kaggle Dataset includes `replay_buffer_latest.pkl` in the `checkpoints/` directory. Without it, the resume will fall back to an empty replay buffer, causing catastrophic forgetting. The buffer file is ~3 GB; verify the dataset upload succeeded.

In [ ]:
# CELL 7: Resume interrupted run — DO NOT RUN on a fresh session
import shutil, os
from src.rl.train import train_sac

PREV_RESULTS    = '/kaggle/input/thesis-rl-checkpoints/results_rl'
CHECKPOINT_PATH = 'results/rl/sac_general_seed0/checkpoints/sac_general_seed0_300000_steps.zip'
SEED = 0

dst = '/kaggle/working/thesis/results/rl'
os.makedirs(dst, exist_ok=True)
shutil.copytree(PREV_RESULTS, dst, dirs_exist_ok=True)
print('Checkpoints restored.')

train_sac(
    total_timesteps=2_000_000, seed=SEED,
    output_dir='/kaggle/working/thesis/results/rl',
    checkpoint_freq=50_000, eval_freq=25_000,
    resume_path=CHECKPOINT_PATH,
    wandb_project='sac-irrigation-thesis',
    verbose=1,
)